# nb_gold_mercado_trabalho — Camada Gold · Mercado de Trabalho Unificado

**Fontes:**
- `silver_cempre` → histórico macro até 2021 (Seção CNAE) · `nb_ingest_cempre_ibge`
- `silver_rais` → estoque micro 2022–2024 (Subclasse CNAE + Porte) · `nb_ingest_rais_bigquery`
- `silver_caged` → fluxo mensal 2020–presente · `nb_ingest_caged`

**Saída:** `gold_mercado_trabalho`  
**Granularidade:** município × ano × (mes) × seção_cnae × subclasse_cnae  
**Modo:** `save_delta()` com V-Order obrigatório (Direct Lake)

**Lógica de período:**
| Período | Fonte | Granularidade |
|---|---|---|
| Até 2021 | CEMPRE | Seção CNAE |
| 2022–2024 | RAIS | Subclasse CNAE + Porte |
| Mensal 2020–presente | CAGED | Mensal · Seção + Subclasse |

In [ ]:
%run ./nb_utils_ibge

In [ ]:
from pyspark.sql.functions import (
    col, lit, trim, sum as spark_sum
)
from pyspark.sql.types import IntegerType, LongType

## 1. Carregar Fontes

In [ ]:
# Cluster map derivado do nb_utils_ibge
cluster_rows = [
    (code, cluster)
    for cluster, codes in CLUSTERS.items()
    for code in codes
]
df_cluster = spark.createDataFrame(cluster_rows, ["id_municipio", "cluster"])

df_cempre = spark.table("silver_cempre")
df_rais   = spark.table("silver_rais")
df_caged  = spark.table("silver_caged")

print(f"[OK] silver_cempre : {df_cempre.count():,} registros")
print(f"[OK] silver_rais   : {df_rais.count():,} registros")
print(f"[OK] silver_caged  : {df_caged.count():,} registros")

## 2. Preparar cada fonte no schema unificado

Schema alvo:
`id_municipio · nome_municipio · cluster · ano · mes · secao_cnae · subclasse_cnae · vinculos_ativos · saldo_mensal · tamanho_estabelecimento · fonte`

In [ ]:
# ── Fonte 1: CEMPRE (até 2021) ──
df_part_cempre = (
    df_cempre
    .filter(col("ano") <= 2021)
    .join(df_cluster, on="id_municipio", how="left")
    .groupBy("id_municipio", "nome_municipio", "cluster", "ano",
             "secao_cnae_cod", "secao_cnae")
    .agg(spark_sum("valor").alias("vinculos_ativos"))
    .select(
        col("id_municipio"),
        col("nome_municipio"),
        col("cluster"),
        col("ano"),
        lit(None).cast(IntegerType()).alias("mes"),
        col("secao_cnae_cod").alias("secao_cnae"),
        lit(None).cast("string").alias("subclasse_cnae"),
        col("vinculos_ativos").cast(LongType()),
        lit(None).cast(LongType()).alias("saldo_mensal"),
        lit(None).cast("string").alias("tamanho_estabelecimento"),
        lit("CEMPRE").alias("fonte")
    )
)

print(f"[OK] CEMPRE (até 2021): {df_part_cempre.count():,} registros")

In [ ]:
# ── Fonte 2: RAIS (2022–2024) ──
# silver_rais já tem nome_municipio e cluster (join validado em 04/05)
df_part_rais = (
    df_rais
    .filter(col("ano") >= 2022)
    .select(
        col("id_municipio"),
        col("nome_municipio"),
        col("cluster"),
        col("ano"),
        lit(None).cast(IntegerType()).alias("mes"),
        col("cnae_2").alias("secao_cnae"),
        col("cnae_2_subclasse").alias("subclasse_cnae"),
        col("quantidade_vinculos_ativos").cast(LongType()).alias("vinculos_ativos"),
        lit(None).cast(LongType()).alias("saldo_mensal"),
        col("tamanho_estabelecimento"),
        lit("RAIS").alias("fonte")
    )
)

print(f"[OK] RAIS (2022+): {df_part_rais.count():,} registros")

In [ ]:
# ── Fonte 3: CAGED — fluxo mensal (nb_ingest_caged · lh_dados_publicos) ──
# silver_caged schema: id_municipio · nome_municipio · cluster · ano · mes ·
#                      secao_cnae · subclasse_cnae · saldo_movimentacao · salario_medio
df_part_caged = (
    df_caged
    .select(
        col("id_municipio"),
        col("nome_municipio"),
        col("cluster"),
        col("ano"),
        col("mes").cast(IntegerType()),
        col("secao_cnae"),
        col("subclasse_cnae"),
        lit(None).cast(LongType()).alias("vinculos_ativos"),
        col("saldo_movimentacao").cast(LongType()).alias("saldo_mensal"),
        lit(None).cast("string").alias("tamanho_estabelecimento"),
        lit("CAGED").alias("fonte")
    )
)

print(f"[OK] CAGED: {df_part_caged.count():,} registros")

## 3. Unificar e Gravar Gold

In [ ]:
df_gold = df_part_cempre.unionByName(df_part_rais).unionByName(df_part_caged)

assert df_gold.count() > 0, "[ERRO] gold_mercado_trabalho vazio antes de gravar"
print(f"[OK] gold_mercado_trabalho: {df_gold.count():,} registros totais")

df_gold.groupBy("fonte").count().orderBy("fonte").show()

In [ ]:
save_delta(df_gold, "gold_mercado_trabalho")
print("[OK] gold_mercado_trabalho gravada com V-Order")

## 4. Validação — Spot Check

In [ ]:
# Cobertura temporal por fonte
print("=== Cobertura por fonte ===")
df_gold.groupBy("fonte").agg(
    {"ano": "min", "ano": "max", "vinculos_ativos": "sum"}
).show()

# Santos + Osasco + Mauá — ano mais recente por fonte
print("\n=== Municípios-âncora ===")
display(
    df_gold
    .filter(col("id_municipio").isin(3548500, 3534401, 3529401))
    .groupBy("nome_municipio", "cluster", "ano", "fonte")
    .agg(spark_sum("vinculos_ativos").alias("total_vinculos"),
         spark_sum("saldo_mensal").alias("saldo_total"))
    .orderBy("nome_municipio", "ano", "fonte")
)